# Puyo Puyo AI — 学習パイプライン（Google Colab GPU）

このノートブックは Google Colab の GPU（T4 / A100）を使って
`puyo-trainer` の学習パイプライン全体を実行し、
学習済みモデルを GitHub にプッシュします。

## 学習フェーズ

| Phase | 内容 | 初回のみ？ |
|-------|------|-----------|
| Phase 1: generate-data | ヒューリスティック AI で棋譜を生成 | ✅ 初回のみ（オプション） |
| Phase 2: train | 教師あり学習でベースモデルを作成 | ✅ 初回のみ（オプション） |
| Phase 3: self-play → train --alphazero | 自己対戦データ生成 → 強化学習 | 🔁 繰り返し実行 |

Phase 1-2 はオプションです。ベースモデルがない場合、self-play はランダム初期化から開始します。
Phase 1-2 で教師あり学習のベースモデルを作成しておくと、Phase 3 の学習効率が向上します。

## 前提条件

1. **GPU ランタイムを選択してください**
   メニュー → `ランタイム` → `ランタイムのタイプを変更` → `T4 GPU`（または A100）

2. **Colab シークレットに `GITHUB_TOKEN` を設定**
   左サイドバー 🔑 → `GITHUB_TOKEN` を追加
   GitHub の Fine-grained PAT、権限: `Contents: Read and write`

## セッション切れ後の再開手順

Colab のセッションは約 90 分（無料）でリセットされます。
セル 2 の keep-alive を実行しておくとアイドルタイムアウトを防げます。
再接続後は以下のセルを順に実行してください：

| セル | 毎回必要か | 補足 |
|------|-----------|------|
| 2: keep-alive | ✅ 推奨 | セッション切断回避（最初に実行） |
| 3: GPU 確認 | ✅ 推奨 | スキップ可 |
| 4: Rust インストール | ✅ 必要 | セッション揮発 |
| 5: リポジトリ | ✅ 必要 | clone/pull |
| 6A: ビルド (Phase 1-2) | 初回のみ | ベースモデルを作る場合 |
| 6B: generate-data | 初回のみ | Phase 1: 棋譜生成 |
| 6C: train | 初回のみ | Phase 2: 教師あり学習 |
| 7: ビルド (Phase 3) | ✅ 必要 | ビルドキャッシュも揮発 |
| 8: self-play | ✅ 必要 | 自己対戦データ生成 → GitHub push |
| 8B: train --alphazero | ✅ 必要 | 強化学習 → GitHub push |

## 継続学習の流れ

セッションをまたいでも、毎回セル 2 → 8B を順番に実行するだけです。
self-play は `artifacts/puyo_model` があれば自動的にロードし、なければランダム初期化で開始します。

In [ ]:
# ============================================================
# セル 2: セッション切断回避（keep-alive）
# Colab のアイドルタイムアウトを防ぐため、
# 定期的に接続ボタンをクリックするスクリプトを実行します。
# ============================================================
from IPython.display import display, Javascript

display(Javascript('''
function keepAlive() {
    const btn = document.querySelector("colab-connect-button");
    if (btn) btn.click();
    console.log("keep-alive: clicked at " + new Date().toLocaleTimeString());
}
setInterval(keepAlive, 60000);
keepAlive();
console.log("keep-alive: started (60s interval)");
'''))

print('セッション keep-alive を開始しました（60秒間隔）')
print('このセルはノートブック実行中ずっと有効です')

In [ ]:
# ============================================================
# セル 3: GPU / CUDA 環境確認
# ============================================================
import subprocess
import glob
import os

print('=== GPU 確認 ===')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('GPU が見つかりません。ランタイムを GPU に変更してください。')
    raise SystemExit('GPU ランタイムが必要です')

print('\n=== CUDA ライブラリ確認 ===')
cuda_libs = (
    glob.glob('/usr/local/cuda*/lib64/libcuda.so*') +
    glob.glob('/usr/lib/x86_64-linux-gnu/libcuda.so*')
)
print('libcuda:', cuda_libs if cuda_libs else '見つかりません')

# burn/cuda-jit が参照する CUDA_PATH を設定
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'

os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print(f'\nCUDA_PATH={CUDA_PATH}')
print('nvcc（コンパイラ）は burn/cuda-jit の JIT 実行には不要です')

In [ ]:
# ============================================================
# セル 4: Rust インストール
# ============================================================
import subprocess
import os
import glob

# rustup でインストール
result = subprocess.run(
    'curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable',
    shell=True, capture_output=False
)

# PATH を永続化
cargo_bin = '/root/.cargo/bin'
os.environ['PATH'] = f"{cargo_bin}:{os.environ['PATH']}"

# CUDA_PATH も再設定（セルをスキップして実行された場合に備えて）
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

# バージョン確認
for cmd in [['rustc', '--version'], ['cargo', '--version']]:
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.strip() if r.returncode == 0 else f'{cmd[0]} が見つかりません')

In [ ]:
# ============================================================
# セル 5: リポジトリ clone / pull
# Colab 左サイドバー 🔑 に GITHUB_TOKEN（Fine-grained PAT）を設定してください
# 必要な権限: Contents (read/write)
# ============================================================
import subprocess
import os
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'
BRANCH = 'develop'

# PAT でクローン（push 時にも認証に使用）
token = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('リポジトリ既存 → git pull')
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, text=True)
    subprocess.run(['git', 'pull'], cwd=REPO_DIR, text=True)
else:
    print(f'リポジトリ clone 中（ブランチ: {BRANCH}）...')
    result = subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], text=True)
    if result.returncode != 0:
        raise RuntimeError('clone 失敗。GITHUB_TOKEN の権限を確認してください。')
    print('clone 完了')

# artifacts ディレクトリを確認
artifacts_dir = os.path.join(REPO_DIR, 'artifacts')
os.makedirs(artifacts_dir, exist_ok=True)
print('\nartifacts/ の内容:')
for f in sorted(os.listdir(artifacts_dir)):
    size = os.path.getsize(os.path.join(artifacts_dir, f))
    print(f'  {f}: {size:,} bytes')

## Phase 1-2: ヒューリスティック AI 学習（初回のみ）

以下のセル 6A〜6C は **初回のみ** 実行してください。  
学習済みモデル (`artifacts/puyo_model.bin`) が既にリポジトリに存在する場合はスキップして、
セル 7 (self-play ビルド) に進んでください。

### 処理の流れ

1. **6A: ビルド** — `generate-data` と `train` バイナリをビルド
2. **6B: generate-data** — ヒューリスティック AI で 10,000 ゲームの棋譜を生成（CPU 処理、時間がかかります）
3. **6C: train** — 生成データで教師あり学習 → モデルを GitHub にプッシュ


In [ ]:
# ============================================================
# セル 6A: ビルド（generate-data + train）
# Phase 1-2 用バイナリを GPU ビルドします。
# ============================================================
import subprocess
import os
import glob

REPO_DIR = '/content/puyopuyo-ai'

# 環境変数を再確認
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print(f'CUDA_PATH={CUDA_PATH}')
print('generate-data + train ビルド開始（初回は時間がかかります）...')

result = subprocess.run(
    ['cargo', 'build', '--release', '-p', 'puyo-trainer',
     '--bin', 'generate-data', '--bin', 'train'],
    cwd=REPO_DIR,
    env=os.environ,
)

if result.returncode == 0:
    for name in ['generate-data', 'train']:
        binary = os.path.join(REPO_DIR, f'target/release/{name}')
        size = os.path.getsize(binary)
        print(f'  {name}: {size:,} bytes')
    print('ビルド成功!')
else:
    print(f'ビルド失敗 (returncode={result.returncode})')


In [ ]:
# ============================================================
# セル 6B: generate-data 実行（Phase 1）
# ヒューリスティック AI で 10,000 ゲームの棋譜を生成します。
# SimulationEvaluator + depth-2 探索のため、CPU のみでも動作しますが
# 所要時間は環境に依存します（30分〜数時間）。
# ============================================================
import subprocess
import os
import time
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'
BRANCH = 'develop'
binary = os.path.join(REPO_DIR, 'target/release/generate-data')

if not os.path.exists(binary):
    raise FileNotFoundError('バイナリが見つかりません。セル 6A のビルドを先に実行してください。')

print('generate-data 開始（10,000 ゲーム）')
print('1,000 ゲームごとに進捗が表示されます')
print('-' * 60)

start_time = time.time()

process = subprocess.Popen(
    [binary],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
elapsed = time.time() - start_time

print('-' * 60)
if process.returncode != 0:
    print(f'generate-data 失敗 (returncode={process.returncode})')
else:
    data_path = os.path.join(REPO_DIR, 'data/training_data.bin')
    size = os.path.getsize(data_path)
    print(f'generate-data 完了！経過時間: {elapsed / 60:.1f} 分')
    print(f'出力: data/training_data.bin ({size:,} bytes)')

    # ============================================================
    # GitHub にプッシュ
    # ============================================================
    token = userdata.get('GITHUB_TOKEN')
    remote_url = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

    subprocess.run(['git', 'config', 'user.email', 'colab-training@example.com'], cwd=REPO_DIR)
    subprocess.run(['git', 'config', 'user.name', 'Colab Training'], cwd=REPO_DIR)
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], cwd=REPO_DIR)

    subprocess.run(['git', 'add', 'data/training_data.bin'], cwd=REPO_DIR)

    timestamp = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
    result = subprocess.run(
        ['git', 'commit', '-m', f'chore: add training data from generate-data [{timestamp}]'],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    print(f'\n{result.stdout.strip()}')
    if result.returncode != 0:
        print('コミット失敗:', result.stderr.strip())
    else:
        print('プッシュ中...')
        result = subprocess.run(
            ['git', 'push', 'origin', BRANCH],
            cwd=REPO_DIR, capture_output=True, text=True
        )
        if result.returncode == 0:
            print('プッシュ完了！')
        else:
            print('プッシュ失敗:')
            print(result.stderr.strip())
            print('→ GITHUB_TOKEN の Contents write 権限を確認してください')

In [ ]:
# ============================================================
# セル 6C: train 実行（Phase 2）→ GitHub 自動プッシュ
# 教師あり学習でベースモデルを作成します。
# 出力: artifacts/puyo_model.bin
# ============================================================
import subprocess
import os
import time
import glob
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'
BRANCH = 'develop'
binary = os.path.join(REPO_DIR, 'target/release/train')

if not os.path.exists(binary):
    raise FileNotFoundError('バイナリが見つかりません。セル 6A のビルドを先に実行してください。')

# CUDA_PATH を再設定
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print('train 開始（教師あり学習）')
print('エポックごとに loss が表示されます')
print('-' * 60)

start_time = time.time()

process = subprocess.Popen(
    [binary],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ,
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
elapsed = time.time() - start_time

print('-' * 60)
if process.returncode != 0:
    print(f'train 失敗 (returncode={process.returncode})')
else:
    model_path = os.path.join(REPO_DIR, 'artifacts/puyo_model.bin')
    size = os.path.getsize(model_path)
    print(f'train 完了！経過時間: {elapsed / 60:.1f} 分')
    print(f'出力: artifacts/puyo_model.bin ({size:,} bytes)')

    # GitHub にプッシュ
    token = userdata.get('GITHUB_TOKEN')
    remote_url = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

    subprocess.run(['git', 'config', 'user.email', 'colab-training@example.com'], cwd=REPO_DIR)
    subprocess.run(['git', 'config', 'user.name', 'Colab Training'], cwd=REPO_DIR)
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], cwd=REPO_DIR)
    subprocess.run(['git', 'add', 'artifacts/puyo_model.bin'], cwd=REPO_DIR)

    timestamp = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
    result = subprocess.run(
        ['git', 'commit', '-m', f'chore: add base model from supervised training [{timestamp}]'],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    print(f'\n{result.stdout.strip()}')
    if result.returncode != 0:
        print('コミット失敗:', result.stderr.strip())
    else:
        print('プッシュ中...')
        result = subprocess.run(
            ['git', 'push', 'origin', BRANCH],
            cwd=REPO_DIR, capture_output=True, text=True
        )
        if result.returncode == 0:
            print('プッシュ完了！')
        else:
            print('プッシュ失敗:', result.stderr.strip())

In [ ]:
# ============================================================
# セル 7: ビルド（GPU）— self-play + train
# 初回は 10〜20 分かかります。Colab の出力に進捗が表示されます。
# ============================================================
import subprocess
import os
import glob

REPO_DIR = '/content/puyopuyo-ai'

# 環境変数を再確認
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ.get('PATH', '')}"
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print(f'CUDA_PATH={CUDA_PATH}')
print('GPU ビルド開始（self-play + train）...')

result = subprocess.run(
    ['cargo', 'build', '--release', '-p', 'puyo-trainer',
     '--bin', 'self-play', '--bin', 'train'],
    cwd=REPO_DIR,
    env=os.environ,
)

if result.returncode == 0:
    for name in ['self-play', 'train']:
        binary = os.path.join(REPO_DIR, f'target/release/{name}')
        size = os.path.getsize(binary)
        print(f'  {name}: {size:,} bytes')
    print('ビルド成功!')
else:
    print(f'\nGPU ビルド失敗 (returncode={result.returncode})')
    print('→ ランタイムが GPU に設定されているか確認してください')

In [ ]:
# ============================================================
# セル 8: self-play 実行 → GitHub 自動プッシュ
# MCTS + NN で自己対戦し、訓練データを生成します。
# モデルがない場合はランダム初期化から開始します。
# ============================================================
import subprocess
import os
import time
import glob
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'
BRANCH = 'develop'
binary = os.path.join(REPO_DIR, 'target/release/self-play')

# ============================================================
# パラメータ設定（ここを変更してください）
# ============================================================
NUM_GAMES = 100             # 自己対戦ゲーム数（デフォルト: 100）
NUM_SIMULATIONS = 200       # MCTS シミュレーション回数/手（デフォルト: 200）
# ============================================================

if not os.path.exists(binary):
    raise FileNotFoundError('バイナリが見つかりません。セル 7 のビルドを先に実行してください。')

# CUDA_PATH を再設定
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print('self-play 開始')
print(f'設定: NUM_GAMES={NUM_GAMES}, NUM_SIMULATIONS={NUM_SIMULATIONS}')
print('-' * 60)

start_time = time.time()

process = subprocess.Popen(
    [binary, '--games', str(NUM_GAMES), '--simulations', str(NUM_SIMULATIONS)],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ,
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
elapsed = time.time() - start_time

print('-' * 60)
if process.returncode != 0:
    print(f'self-play 失敗 (returncode={process.returncode})')
else:
    data_path = os.path.join(REPO_DIR, 'data/alphazero_data.bin')
    size = os.path.getsize(data_path)
    print(f'self-play 完了！経過時間: {elapsed / 60:.1f} 分')
    print(f'出力: data/alphazero_data.bin ({size:,} bytes)')

    # GitHub にプッシュ
    token = userdata.get('GITHUB_TOKEN')
    remote_url = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

    subprocess.run(['git', 'config', 'user.email', 'colab-training@example.com'], cwd=REPO_DIR)
    subprocess.run(['git', 'config', 'user.name', 'Colab Training'], cwd=REPO_DIR)
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], cwd=REPO_DIR)
    subprocess.run(['git', 'add', 'data/alphazero_data.bin'], cwd=REPO_DIR)

    timestamp = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
    result = subprocess.run(
        ['git', 'commit', '-m', f'chore: add self-play data [{timestamp}]'],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    print(f'\n{result.stdout.strip()}')
    if result.returncode != 0:
        print('コミット失敗:', result.stderr.strip())
    else:
        print('プッシュ中...')
        result = subprocess.run(
            ['git', 'push', 'origin', BRANCH],
            cwd=REPO_DIR, capture_output=True, text=True
        )
        if result.returncode == 0:
            print('プッシュ完了！')
        else:
            print('プッシュ失敗:', result.stderr.strip())

In [ ]:
# ============================================================
# セル 8B: train --alphazero 実行 → GitHub 自動プッシュ
# self-play データで強化学習（Policy CE + Value MSE with Value Transform）
# 出力: artifacts/puyo_model.bin（更新済みモデル）
# ============================================================
import subprocess
import os
import time
import glob
from google.colab import userdata

REPO_DIR = '/content/puyopuyo-ai'
BRANCH = 'develop'
binary = os.path.join(REPO_DIR, 'target/release/train')

if not os.path.exists(binary):
    raise FileNotFoundError('バイナリが見つかりません。セル 7 のビルドを先に実行してください。')

# CUDA_PATH を再設定
cuda_dirs = sorted(glob.glob('/usr/local/cuda*'))
CUDA_PATH = cuda_dirs[-1] if cuda_dirs else '/usr/local/cuda'
os.environ['CUDA_PATH'] = CUDA_PATH
os.environ['CUDA_HOME'] = CUDA_PATH

print('train --alphazero 開始（強化学習）')
print('エポックごとに policy_loss / value_loss が表示されます')
print('-' * 60)

start_time = time.time()

process = subprocess.Popen(
    [binary, '--alphazero'],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ,
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
elapsed = time.time() - start_time

print('-' * 60)
if process.returncode != 0:
    print(f'train --alphazero 失敗 (returncode={process.returncode})')
else:
    model_path = os.path.join(REPO_DIR, 'artifacts/puyo_model.bin')
    size = os.path.getsize(model_path)
    print(f'train --alphazero 完了！経過時間: {elapsed / 60:.1f} 分')
    print(f'出力: artifacts/puyo_model.bin ({size:,} bytes)')

    # GitHub にプッシュ
    token = userdata.get('GITHUB_TOKEN')
    remote_url = f'https://{token}@github.com/hfappmaker/puyopuyo-ai.git'

    subprocess.run(['git', 'config', 'user.email', 'colab-training@example.com'], cwd=REPO_DIR)
    subprocess.run(['git', 'config', 'user.name', 'Colab Training'], cwd=REPO_DIR)
    subprocess.run(['git', 'remote', 'set-url', 'origin', remote_url], cwd=REPO_DIR)
    subprocess.run(['git', 'add', 'artifacts/puyo_model.bin'], cwd=REPO_DIR)

    timestamp = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
    result = subprocess.run(
        ['git', 'commit', '-m', f'chore: update model from alphazero training [{timestamp}]'],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    print(f'\n{result.stdout.strip()}')
    if result.returncode != 0:
        print('コミット失敗:', result.stderr.strip())
    else:
        print('プッシュ中...')
        result = subprocess.run(
            ['git', 'push', 'origin', BRANCH],
            cwd=REPO_DIR, capture_output=True, text=True
        )
        if result.returncode == 0:
            print('プッシュ完了！')
        else:
            print('プッシュ失敗:', result.stderr.strip())

## 完了！

セル 8B が正常終了すると、モデルが自動的に GitHub の `develop` ブランチにプッシュされます。

### GitHub にプッシュされるファイル

| ファイル | 内容 | Phase |
|---------|------|-------|
| `artifacts/puyo_model.bin` | 学習済みモデル（Phase 2 で作成、Phase 3 で更新） | 2, 3 |
| `data/alphazero_data.bin` | self-play で生成した訓練データ | 3 |

### 学習パイプラインの全体像

```
Phase 1: generate-data（オプション）
  → data/training_data.bin（棋譜データ）

Phase 2: train（オプション）
  → artifacts/puyo_model.bin（ベースモデル）

Phase 3: self-play → train --alphazero（繰り返し実行）
  → data/alphazero_data.bin（self-play データ）
  → artifacts/puyo_model.bin（強化モデル）
  ※ モデルがない場合はランダム初期化から開始
  ※ Value Transform（MuZero方式）で MSE 損失を安定化
  ※ コンテキストに残り手数を含め、Value Head の精度を向上
```

### 継続学習の流れ

セッションをまたいで学習を継続する場合は、毎回セル 3 → 8B を順番に実行するだけです。
`self-play` は `artifacts/puyo_model` があれば自動的にロードし、なければランダム初期化で開始します。
`train --alphazero` も同様に、既存モデルがあればファインチューニング、なければ新規学習します。

### 次のステップ（ローカルで作業する場合）

1. `git pull origin develop` でモデルを取得
2. WASM を再ビルド:
   ```bash
   bash scripts/build-wasm.sh
   ```